In [ ]:
import os, sys, importlib.util, pandas as pd
import joblib
from pathlib import Path
from plttools import *
import utils

spec = importlib.util.spec_from_file_location("ns", "number_samples.py")
ns = importlib.util.module_from_spec(spec)
spec.loader.exec_module(ns)

LOAD = True
SAVE_FIGS = False
RESULTS_DIR = None

ns.LOAD = LOAD
ns.SAVE = not LOAD
if RESULTS_DIR is not None:
    ns.PATH = str(RESULTS_DIR) + os.sep
os.makedirs(ns.PATH, exist_ok=True)
print("Results folder", ns.PATH)


Results folder /home/nav/projects/repos/nomad/results/nonneg_colide/samples/


In [12]:
n_dags = 100
n_samples_values = np.array([50, 60, 80, 100, 200, 500, 1000, 5000, 10000])
thr = .2
verb = True

Exps = ns.build_experiments()
scenarios = {sc["name"]: sc for sc in ns.build_scenarios(n_nodes=20)}

pd.DataFrame([{"leg": e["leg"], "model": e["model"].__name__, "sigma": e["sigma"], "adapt_lamb": e["adapt_lamb"],
               **e.get("init", {}), **e["args"]} for e in Exps]).set_index("leg")


,model,sigma,adapt_lamb,primal_opt,acyclicity,equal_var,stepsize,step_type,alpha_0,rho_0,...,iters_in,iters_out,beta,sca_adam,loss_type,lambda1,T,warm_iter,max_iter,lr
leg,,,,,,,,,,,,,,,,,,,,,
NN-CoLiDE-NV,MetMulColide,True,True,sca,logdet,False,0.0003,fixed,0.01,0.05,...,30000.0,10.0,2.0,True,NaN,NaN,NaN,NaN,NaN,NaN
NN-CoLiDE-EV,MetMulColide,True,True,sca,logdet,True,0.0003,fixed,0.01,0.05,...,30000.0,10.0,2.0,True,NaN,NaN,NaN,NaN,NaN,NaN
NOMAD-adam,MetMulDagma,False,True,adam,logdet,NaN,0.0050,fixed,0.10,0.10,...,5000.0,10.0,1.5,NaN,NaN,NaN,NaN,NaN,NaN,NaN
DAGMA,DAGMA_linear,False,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,l2,0.05,4.0,20000.0,70000.0,0.0003
CoLiDE-EV,colide_ev,True,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,0.05,4.0,20000.0,70000.0,0.0003
CoLiDE-NV,colide_nv,True,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,0.05,4.0,20000.0,70000.0,0.0003


In [ ]:
results = {}


def run_or_load(name, skip=()):
    sc = scenarios[name]
    out = ns.run_or_load_samples_results(sc["data_params"], n_samples_values, Exps, n_dags, sc["name"], thr=thr, verb=verb)
    *metrics, exps_out, xvals = out
    results[name] = {"metrics": dict(zip(ns.METRIC_NAMES, metrics)), "exps": exps_out, "xvals": xvals, "scenario": sc}
    return results[name]


def show_scenario(name, skip=()):
    res = run_or_load(name)
    m, exps_out, xvals, sc = res["metrics"], res["exps"], res["xvals"], res["scenario"]
    skip = list(skip)
    prefix = ns.samples_results_prefix(sc["name"])

    # SHD (mean +- std) and Fro error (median, prctiles 25-75)
    fig, _ = utils.plot_shd_error_pair(m["shd"], m["err"], exps_out, xvals, xlabel="Number of samples",
                                       shd_ylabel="SHD", err_ylabel="Fro Error", skip_idx=skip, alpha=.25,
                                       shd_agg="mean", shd_deviation="std", err_agg="median", err_deviation="prctile",
                                       err_plot_func="loglog", figsize=(10, 5), title=sc["title"])
    if SAVE_FIGS:
        fig.savefig(f"{prefix}_summary.png", bbox_inches="tight")

    # Sigma error (only methods that estimate Sigma)
    fig, _ = ns.plot_sigma_error(m["err_sig"], exps_out, xvals, skip_idx=skip, title=f'{sc["title"]} - Sigma error')
    if SAVE_FIGS:
        fig.savefig(f"{prefix}_sigma_error.png", bbox_inches="tight")

    # Tables at each number of samples
    legs = [e["leg"] for e in exps_out]
    for metric, agg, label in [("shd", np.mean, "SHD (mean)"), ("err", np.median, "Fro error (median)"),
                               ("err_sig", np.nanmedian, "Sigma error (median)"), ("runtime", np.mean, "time (mean, s)")]:
        with np.errstate(all="ignore"):
            table = pd.DataFrame(agg(m[metric], axis=0), index=xvals, columns=legs).rename_axis("samples")
        print(f"\n{sc['title']} - {label}")
        display(table.round(4))
    return res


def show_all_metrics(name, agg="mean", skip=()):
    m, exps_out, xvals = results[name]["metrics"], results[name]["exps"], results[name]["xvals"]
    utils.plot_all_metrics(m["shd"], m["tpr"], m["fdr"], m["fscore"], m["err"], m["acyc"], m["runtime"],
                           m["dag_count"], xvals, exps_out, skip_idx=list(skip), agg=agg)


In [ ]:
skip = []
res = show_scenario("er4_N100_var1", skip=skip)


In [ ]:
show_all_metrics("er4_N100_var1", agg="mean", skip=skip)
show_all_metrics("er4_N100_var1", agg="median", skip=skip)
